# Deadlock All-Chat — High-Throughput Replay Chat Collector & Username Query Engine

Collect **all in-game chat messages (both All-Chat and Team-Chat)** from Deadlock match replays (`https://api.deadlock-api.com/`) and query any player's chat history by username in milliseconds.

### Why this architecture is optimal for Google Colab
1. **Zero-Entity-Decoding Rust Streaming Parser (`deadlock-chat-extractor`)**:
   - Standard Deadlock demos are 50–450 MB uncompressed (~30–100 MB compressed). Valve's CDN serves newer `.dem.bz2` files with **Zstandard** compression (~300+ MB/s decompression) and older files with **Bzip2** — our extractor sniffs magic bytes automatically.
   - Instead of writing uncompressed `.dem` files to Colab's limited disk or decoding entity deltas (`svc_PacketEntities`, ~98% of replay bytes), our Rust binary **streams HTTP chunks straight through Zstd/Bzip2 → PBDEMS2 frame scanner**, reading only `userinfo` string tables, `CCitadelUserMsg_ChatMsg` (ID 314), and `CCitadelUserMsg_PostMatchDetails` (ID 316).
   - **~30× faster CPU parsing** than full entity parsers (`boon-deadlock`), **0 MB of temporary uncompressed disk I/O**, and runs **16–32 concurrent HTTP streams** across Valve's ~68 replay clusters to saturate Colab's network bandwidth.
2. **Use a Standard CPU Runtime (No GPU needed)**:
   - Replay processing is purely **network I/O + Zstd/Snappy CPU**, so a free/Pro **CPU** runtime (or High-RAM CPU) is ideal and avoids GPU quota limits.
3. **Crash-Safe Parquet Data Lake on Google Drive + DuckDB**:
   - Extracted chat messages are ~1–2 KB/match in Zstd Parquet (**1,000,000 matches ≈ 1–2 GB**).
   - Every chunk writes an immutable Parquet shard + processed-match checkpoint so interrupted Colab sessions resume cleanly with zero duplicate downloads.
4. **Hybrid Collection + On-Demand Username Lookup**:
   - **Bulk Mode**: continuously harvest recent matches from Deadlock API's public `match_salts` Parquet lake.
   - **Username Lookup Mode**: query any Steam username (current or historical) across your local lake **and** optionally live-fetch & parse any uncached recent matches for that player in seconds!

## 1. Setup: Mount Google Drive & Install Dependencies
Run this cell once per Colab session. The compiled Rust binary is cached on Google Drive so future sessions start in ~5 seconds.

In [ ]:
#@title Mount Google Drive & Configure Paths { display-mode: "form" }
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
REPO_URL = "https://github.com/veroiuxl/deadlock-all-chat.git" #@param {type:"string"}
DRIVE_DATA_DIR = "/content/drive/MyDrive/deadlock_all_chat" #@param {type:"string"}

import importlib, os, pathlib, shutil, subprocess, sys

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = pathlib.Path(DRIVE_DATA_DIR)
else:
    DATA_DIR = pathlib.Path("/content/deadlock_all_chat")

DATA_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR = pathlib.Path("/content/deadlock-all-chat")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

# Install package into the active kernel Python (non-editable avoids Colab .pth restart requirement)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_DIR)], check=True)
src_dir = str(REPO_DIR / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
importlib.invalidate_caches()
# Evict any previously imported modules so re-running Setup after a git pull loads fresh code
for _m in [m for m in list(sys.modules) if m == "deadlock_chat" or m.startswith("deadlock_chat.")]:
    del sys.modules[_m]

import deadlock_chat
print(f"Imported deadlock_chat from {deadlock_chat.__file__}")

# Versioned binary cache on Google Drive (rebuilds once per extractor release, ~60-90s; ~2s thereafter)
EXTRACTOR_VERSION = "v0.2"
DRIVE_BIN = DATA_DIR / "bin" / f"deadlock-chat-extractor-{EXTRACTOR_VERSION}"
LOCAL_BIN = pathlib.Path("/tmp/deadlock-chat-extractor")

if DRIVE_BIN.is_file():
    shutil.copy2(DRIVE_BIN, LOCAL_BIN)
    LOCAL_BIN.chmod(0o755)
    print(f"Loaded cached Rust extractor ({EXTRACTOR_VERSION}) from {DRIVE_BIN}")
else:
    if shutil.which("cargo") is None:
        print("Installing Rust toolchain (one-time build; binary will be cached to Drive)...")
        subprocess.run(
            "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal",
            shell=True,
            check=True,
        )
        os.environ["PATH"] = f"{pathlib.Path.home() / '.cargo' / 'bin'}:{os.environ['PATH']}"

    print(f"Compiling deadlock-chat-extractor {EXTRACTOR_VERSION} (release mode)...")
    subprocess.run(
        ["cargo", "build", "--release", "--manifest-path", str(REPO_DIR / "rust/chat-extractor/Cargo.toml")],
        check=True,
    )
    built = REPO_DIR / "rust/chat-extractor/target/release/deadlock-chat-extractor"
    shutil.copy2(built, LOCAL_BIN)
    LOCAL_BIN.chmod(0o755)
    DRIVE_BIN.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(built, DRIVE_BIN)
    print(f"Compiled and cached binary to {DRIVE_BIN}")

os.environ["DEADLOCK_CHAT_BIN"] = str(LOCAL_BIN)
print(f"Ready! Data lake directory: {DATA_DIR}")

## 2. Bulk Corpus Harvester (Continuous Streaming + Multi-Worker Sharding)
Streams matches through a single long-lived Rust worker pool with **round-robin per-cluster scheduling** (no chunk straggler barriers!).

- **Zero straggler stalls:** Freed worker slots refill immediately while Parquet shards flush to Drive every `FLUSH_EVERY` completed matches (via fast `/tmp` staging).
- **Safe to stop at any time:** Clicking Colab's Stop button flushes all already-completed matches before exiting.
- **Scale across multiple Colab tabs:** Set `NUM_WORKERS = 3` and open 3 tabs with `WORKER_ID = 0`, `1`, `2` pointing at the same `DRIVE_DATA_DIR` — each tab harvests disjoint `match_id % NUM_WORKERS` slices with zero overlap!

In [ ]:
#@title Run High-Throughput Bulk Chat Harvest { display-mode: "form" }
TARGET_MATCHES = 2000 #@param {type:"integer"}
CONCURRENCY = 64 #@param {type:"integer"}
MAX_PER_HOST = 6 #@param {type:"integer"}
FLUSH_EVERY = 250 #@param {type:"integer"}
DAYS_BACK = 10 #@param {type:"integer"}
SOURCE = "lake" #@param ["lake", "live"]
WORKER_ID = 0 #@param {type:"integer"}
NUM_WORKERS = 1 #@param {type:"integer"}
API_KEY = "" #@param {type:"string"}

import json, sys
if "/content/deadlock-all-chat/src" not in sys.path:
    sys.path.insert(0, "/content/deadlock-all-chat/src")
from deadlock_chat import ChatLake, harvest_bulk

lake = ChatLake(DATA_DIR)
summary = harvest_bulk(
    lake=lake,
    target_matches=TARGET_MATCHES,
    flush_every=FLUSH_EVERY,
    concurrency=CONCURRENCY,
    max_per_host=MAX_PER_HOST,
    days_back=DAYS_BACK,
    source=SOURCE,
    worker_id=WORKER_ID,
    num_workers=NUM_WORKERS,
    api_key=API_KEY or None,
    compact_every_flushes=10,
    use_local_salts_cache=True,
    verbose=True,
)
print("\nRun summary:")
print(json.dumps(summary, indent=2))

## 3. Query a Username (Instant Lake Query + Optional Live Fetch)
Enter any Steam username to retrieve all chat messages they typed.

- **`LIVE_FETCH_UNCACHED = True`**: automatically resolves their Steam `account_id` via Deadlock API, downloads & parses any of their recent replays (last 21 days) not yet in your lake in parallel, saves ALL players' messages from those games to your Drive lake, and displays their full chat log.
- **`INCLUDE_HISTORICAL_NAMES = True`**: also searches Deadlock API's `steam_profile_observed_names` Parquet table (6.1M rows) to catch players who renamed their Steam account.

In [ ]:
#@title Query Player Chat Messages by Username { display-mode: "form" }
USERNAME = "" #@param {type:"string"}
EXACT_MATCH = False #@param {type:"boolean"}
LIVE_FETCH_UNCACHED = True #@param {type:"boolean"}
INCLUDE_HISTORICAL_NAMES = False #@param {type:"boolean"}
CHAT_TYPE = "any" #@param ["any", "all", "team"]
MAX_LIVE_MATCHES = 30 #@param {type:"integer"}
CONCURRENCY = 16 #@param {type:"integer"}
LIMIT = 500 #@param {type:"integer"}

import polars as pl
from IPython.display import display
import sys
if "/content/deadlock-all-chat/src" not in sys.path:
    sys.path.insert(0, "/content/deadlock-all-chat/src")
from deadlock_chat import ChatLake, lookup_player_chats

if not USERNAME.strip():
    print("Enter a Steam username in the form field above and re-run this cell.")
else:
    lake = ChatLake(DATA_DIR)
    res = lookup_player_chats(
        lake=lake,
        username=USERNAME.strip(),
        exact=EXACT_MATCH,
        live_fetch=LIVE_FETCH_UNCACHED,
        include_historical_names=INCLUDE_HISTORICAL_NAMES,
        max_live_matches=MAX_LIVE_MATCHES,
        concurrency=CONCURRENCY,
        chat_type=None if CHAT_TYPE == "any" else CHAT_TYPE,
        limit=LIMIT,
        verbose=True,
    )
    df = res["messages"]
    print(f"\nFound {len(df):,} chat message(s) for {USERNAME!r} (live-fetched {res['live_fetched_matches']} new match(es)).")
    if not df.is_empty():
        formatted = df.with_columns(
            pl.from_epoch("start_time", time_unit="s").alias("match_utc"),
            (
                (pl.col("game_time_s").cast(pl.Int32) // 60).cast(pl.Utf8).str.zfill(2)
                + ":"
                + (pl.col("game_time_s").cast(pl.Int32) % 60).cast(pl.Utf8).str.zfill(2)
            ).alias("clock"),
        ).select(["match_utc", "match_id", "clock", "player_name", "account_id", "hero_id", "chat_type", "text"])
        display(formatted.to_pandas())

## 4. Full-Text Search, Corpus Stats & Shard Compaction
Query across all collected players using DuckDB SQL or keyword search, and periodically compact Parquet shards for optimal query speed.

In [ ]:
#@title Search Chat Text & View Corpus Statistics { display-mode: "form" }
KEYWORD = "gg" #@param {type:"string"}
RUN_COMPACT = False #@param {type:"boolean"}

import json
from IPython.display import display
import sys
if "/content/deadlock-all-chat/src" not in sys.path:
    sys.path.insert(0, "/content/deadlock-all-chat/src")
from deadlock_chat import ChatLake

lake = ChatLake(DATA_DIR)

if RUN_COMPACT:
    print("Compacting Parquet shards (deduplicating & sorting by account_id)...")
    print(json.dumps(lake.compact(), indent=2))

print("=== Corpus Statistics ===")
print(json.dumps(lake.stats(), indent=2))

if KEYWORD.strip():
    hits = lake.search_messages(KEYWORD.strip(), limit=50)
    print(f"\n=== Top {len(hits)} messages containing {KEYWORD!r} ===")
    if not hits.is_empty():
        display(hits.select(["match_id", "game_time_s", "player_name", "chat_type", "text"]).to_pandas())

## 5. Optional: Benchmark Rust Streaming Extractor vs `boon-deadlock`
Compare our zero-entity-decoding Rust parser directly against `boon-deadlock` on a real Deadlock replay right inside Colab.

In [ ]:
#@title Run Live Parser Benchmark { display-mode: "form" }
!pip install -q boon-deadlock

import time, urllib.request, pathlib
import sys
if "/content/deadlock-all-chat/src" not in sys.path:
    sys.path.insert(0, "/content/deadlock-all-chat/src")
from deadlock_chat.api import DeadlockApiClient
from deadlock_chat.extractor import extract_file_rust, extract_file_boon

with DeadlockApiClient() as api:
    jobs = api.fetch_recent_replay_jobs_from_lake(days_back=3, limit=5)

sample_path = pathlib.Path("/tmp/bench_sample.dem.bz2")
downloaded_job = None
for j in jobs:
    try:
        urllib.request.urlretrieve(j.demo_url, sample_path)
        downloaded_job = j
        break
    except Exception:
        continue

if downloaded_job:
    sz_mb = sample_path.stat().st_size / 1e6
    print(f"Benchmarking match {downloaded_job.match_id} ({sz_mb:.1f} MB compressed)...")

    t0 = time.perf_counter()
    df_rust = extract_file_rust(sample_path, match_id=downloaded_job.match_id)
    t_rust = time.perf_counter() - t0

    t1 = time.perf_counter()
    df_boon = extract_file_boon(sample_path, match_id=downloaded_job.match_id)
    t_boon = time.perf_counter() - t1

    print(f"Rust streaming (0 temp disk): {t_rust*1000:7.1f} ms | {len(df_rust)} chat msgs")
    print(f"boon-deadlock (disk mmap):    {t_boon*1000:7.1f} ms | {len(df_boon)} chat msgs")
    print(f"Speedup:                      {t_boon / max(t_rust, 1e-6):.1f}x faster")
    assert df_rust["text"].to_list() == df_boon["text"].to_list(), "Chat text mismatch!"
    print("Exact parity verified: Rust output matches boon-deadlock 100%!")
else:
    print("Could not download a live sample replay; try again.")